In [49]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer, util


!pip install datasets -q
!pip install transformers -q
!pip install sentence-transformers


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index = False)

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [4]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [5]:
print(train.shape)
print(test.shape)

print(train.columns)

(2000, 8)
(500, 7)
Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


# Milestone 1


Milestone 1 Question 1

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [8]:
# Milestone 1 Q1
freq = train["answer"].value_counts()

print(freq)

answer_q1 = freq.max() + freq.min()

print("M1Q1 Answer =", answer_q1)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
M1Q1 Answer = 814


Milestone 1 Question 2

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [9]:
# Milestone 1 Q2

all_prompts = " ".join(
    train["prompt"].astype(str)
)

cleaned = (
    all_prompts
    .lower()
    .translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

words = cleaned.split()

vocab_size = len(set(words))

print("M1Q2 Answer =", vocab_size)

M1Q2 Answer = 859


Milestone 1 Question 3

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [10]:
# Milestone 1 Q3
prompt = train.loc[0, "prompt"]

cleaned = (
    prompt.lower()
    .translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

tokens = cleaned.split()

filtered_tokens = [
    word
    for word in tokens
    if word not in ENGLISH_STOP_WORDS
]

print(filtered_tokens)
print("M1Q3 Answer =", len(filtered_tokens))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
M1Q3 Answer = 13


Milestone 1 Question 4 

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [11]:
# Milestone 1 Q4
combined_text = []

for _, row in train.iterrows():

    text = (
        str(row["prompt"]) + " "
        + str(row["A"]) + " "
        + str(row["B"]) + " "
        + str(row["C"]) + " "
        + str(row["D"]) + " "
        + str(row["E"])
    )

    combined_text.append(text)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(
    combined_text
)

print("M1Q4 Answer =", X.shape[1])

M1Q4 Answer = 2762


Milestone 1 Question 5

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places). 

In [12]:
# Milestone 1 Q5
prompt = train.loc[0, "prompt"]

option_a = train.loc[0, "A"]

prompt_vec = vectorizer.transform(
    [prompt]
)

option_vec = vectorizer.transform(
    [option_a]
)

similarity = cosine_similarity(
    prompt_vec,
    option_vec
)[0][0]

print(
    "M1Q5 Answer =",
    round(similarity, 4)
)

M1Q5 Answer = 0.272


Milestone 1 Question 6

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [13]:
# Milestone 1 Q6

correct = 0

for _, row in train.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform(
        [prompt]
    )

    similarities = {}

    for option_name, option_text in options.items():

        option_vec = vectorizer.transform(
            [option_text]
        )

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[option_name] = sim

    prediction = max(
        similarities,
        key=similarities.get
    )

    if prediction == row["answer"]:
        correct += 1

accuracy = (
    correct
    / len(train)
) * 100

print(
    "Q6 Answer =",
    round(accuracy, 2)
)

Q6 Answer = 13.55


Milestone 1 Question 7

If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B ?  


ANSWER: 
Ground Truth Answer: C

Predicted Ranking: [C, A, B]

The correct answer appears at Rank 1.

For a single relevant answer, Average Precision at 3 (AP@3) is calculated as:
[
AP@3 = 1/ Rank
]

Therefore,
[
AP@3 = 1/1 = 1.0
]

Final Answer: 1.0

Milestone 1 Question 8

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  


Answer:

Ground Truth Answer: **B**


Predicted Ranking: **[D, B, E]**


The correct answer appears at **Rank 2**.


For a single relevant answer, Average Precision at 3 (AP@3) is calculated as:


[
AP@3 = 1/Rank
]


Therefore,

[
AP@3 = 1/2 = 0.5
]


**Final Answer: 0.5**


Milestone 1 Question 9 

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [14]:
# Milestone 1 Q9

freq = train["answer"].value_counts()

print(freq)

# Top 3 most frequent answers
top3 = list(freq.index[:3])

print("Top 3 answers:", top3)

def map3(actual, preds):
    for i, p in enumerate(preds):
        if p == actual:
            return 1/(i+1)
    return 0

scores = []

for ans in train["answer"]:
    scores.append(map3(ans, top3))

baseline_map3 = sum(scores) / len(scores)

print("MAP@3 =", baseline_map3)
print("Rounded =", round(baseline_map3, 4))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3 answers: ['B', 'C', 'A']
MAP@3 = 0.42125
Rounded = 0.4213


Milestone 1 Question 10

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set? 

In [15]:
# Milestone 1 Q10

combined_text = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

TfidfVectorizer(stop_words='english')

In [16]:
# Milestone 1 Q10

def map3(actual, preds):
    for i, p in enumerate(preds):
        if p == actual:
            return 1/(i+1)
    return 0

scores = []
tfidf_predictions = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[option] = sim

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]
    
    tfidf_predictions.append(top3)

    score = map3(
        row["answer"],
        top3
    )

    scores.append(score)

final_map3 = sum(scores) / len(scores)

print("MAP@3 =", final_map3)
print("Rounded =", round(final_map3, 4))

MAP@3 = 0.2961666666666667
Rounded = 0.2962


In [17]:
print(len(tfidf_predictions))
print(tfidf_predictions[:5])

2000
[['C', 'D', 'B'], ['C', 'B', 'A'], ['A', 'B', 'C'], ['C', 'D', 'B'], ['D', 'E', 'B']]


# Milestone 2


Milestone 2 Question 1

Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [18]:
# Milestone 2 Q1
train_dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

In [19]:
# Milestone 2 Q1
def create_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

In [20]:
# Milestone 2 Q1
train_dataset = train_dataset.map(create_combined_text)

In [21]:
# Milestone 2 Q1
answer = len(train_dataset[51]["combined_text"])

print("Answer:", answer)

Answer: 614


In [22]:
print(train_dataset[1])

{'id': 2, 'prompt': 'What is accelerator-based light-ion fusion?', 'A': 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'B': 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'C': 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce

Milestone 2 Question 2

Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  


In [23]:
# Milestone 2 Q2

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary Size =", tokenizer.vocab_size)

vocab.txt: 0.00B [00:00, ?B/s]

Vocabulary Size = 30522


Milestone 2 Question 3

Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [24]:
# Milestone 2 Q3

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


Milestone 2 Question 4

Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 
What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [25]:
# Milestone 2 Q4

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert the Hugging Face Column to a Python list
prompts = list(train_dataset["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [26]:
print(type(prompts))
print(type(prompts[0]))
print(prompts[0])

<class 'list'>
<class 'str'>
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


Milestone 2 Question 5

BERT/RoBERTa Architecture & Attention Mechanisms

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 
In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

Milestone 2 Q5 Solution


In the BERT Base architecture, the hidden embedding size is 768 and the model uses 12 attention heads. Since the hidden representation is divided equally among all attention heads, the size of each attention head is calculated as:

Head Dimension = Hidden Size / Number of Heads = 768 / 12 = 64

Therefore, each attention head has a dimensionality of 64.

Milestone 2 Question 6

Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 
What is the exact shape of the last_hidden_state tensor returned? 
Note: We follow zero-indexing here.

In [27]:
# Milestone 2  Q6

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load BERT model
model = AutoModel.from_pretrained("bert-base-uncased")

# First prompt (row ID 0)
text = train_dataset[0]["prompt"]

print(text)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [28]:
inputs = tokenizer(
    text,
    return_tensors="pt"
)
inputs

{'input_ids': tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])}

In [29]:
outputs = model(**inputs)
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.4677, -0.0754, -0.2019,  ..., -0.0726,  0.6572,  0.5506],
         [-0.3783, -0.0790, -0.0184,  ...,  0.0546,  0.4026,  0.1284],
         [-1.1947, -0.7359,  0.2192,  ...,  0.6096,  1.0815,  0.1809],
         ...,
         [-0.2402, -0.4758, -0.2410,  ...,  0.0362,  0.9403,  0.3462],
         [-0.3097, -0.7685, -0.4455,  ...,  0.3885,  0.8686, -0.3180],
         [ 0.6300,  0.2221, -0.1082,  ...,  0.4051, -0.2192,  0.0341]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.8721, -0.4912, -0.9036,  0.7628,  0.6580, -0.1722,  0.8342,  0.2401,
         -0.8083, -1.0000, -0.5096,  0.9392,  0.9778,  0.5440,  0.8579, -0.6980,
         -0.2626, -0.6476,  0.3560, -0.2397,  0.7059,  1.0000,  0.0933,  0.3548,
          0.4943,  0.9899, -0.7505,  0.8854,  0.9418,  0.6376, -0.5243,  0.2739,
         -0.9872, -0.2204, -0.8876, -0.9957,  0.4414, -0.7344,  0.0493, -0.0066,
         -0.8635,  0.2816,  1.00

In [30]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [31]:
outputs.last_hidden_state

tensor([[[-0.4677, -0.0754, -0.2019,  ..., -0.0726,  0.6572,  0.5506],
         [-0.3783, -0.0790, -0.0184,  ...,  0.0546,  0.4026,  0.1284],
         [-1.1947, -0.7359,  0.2192,  ...,  0.6096,  1.0815,  0.1809],
         ...,
         [-0.2402, -0.4758, -0.2410,  ...,  0.0362,  0.9403,  0.3462],
         [-0.3097, -0.7685, -0.4455,  ...,  0.3885,  0.8686, -0.3180],
         [ 0.6300,  0.2221, -0.1082,  ...,  0.4051, -0.2192,  0.0341]]],
       grad_fn=<NativeLayerNormBackward0>)

Milestone 2 Question 7

Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places). 

In [32]:
# Milestone 2 Q7

cls_embedding = outputs.last_hidden_state[0, 0, :]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

-1.2001


Milestone 2 Question 8

Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 
What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places). 

In [33]:
# Milestone 2 Q8

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:")
for i, token in enumerate(tokens):
    print(i, token)

attention = outputs.attentions[-1][0][0]

fusion_index = tokens.index("fusion")

print("\nFusion index:", fusion_index)

weight = attention[0, fusion_index].item()

print("Attention weight:", round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]

Fusion index: 4
Attention weight: 0.1025


Milestone 2 Question 9

Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.


In [34]:
# Milestone 2 Q9

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_embedding)

print("Cosine Similarity:", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine Similarity: 0.7658


Milestone 2 Question 10

Build two complete ranking pipelines evaluating every row in train.csv.
Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.
Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.
First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 
Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count? 


In [35]:
# Milestone 2 Q10

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
# Milestone 2 Q10

def map3_score(true_answer, prediction):

    if true_answer == prediction[0]:
        return 1

    elif true_answer == prediction[1]:
        return 1/2

    elif true_answer == prediction[2]:
        return 1/3

    return 0

In [37]:
# Milestone 2 Q10

labels = ["A","B","C","D","E"]

mini_scores = []
mini_predictions = []

for row in train_dataset:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    similarities = []

    for option in labels:

        option_embedding = model.encode(
            row[option],
            convert_to_tensor=True
        )

        score = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()

        similarities.append(score)

    ranking = sorted(
        zip(labels, similarities),
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in ranking[:3]]

    mini_predictions.append(top3)

    mini_scores.append(
        map3_score(
            row["answer"],
            top3
        )
    )

print(len(mini_predictions))
print(sum(mini_scores)/len(mini_scores))

2000
0.4230833333333333


In [38]:
# Milestone 2 Q10

count = 0

for i,row in enumerate(train_dataset):

    answer = row["answer"]

    tfidf_top3 = tfidf_predictions[i]

    mini_top3 = mini_predictions[i]

    if (

        answer not in tfidf_top3

        and

        answer in mini_top3

    ):

        count += 1

print(count)

502


In [39]:
print("Train dataset:", len(train_dataset))
print("TF-IDF predictions:", len(tfidf_predictions))
print("MiniLM predictions:", len(mini_predictions))

Train dataset: 2000
TF-IDF predictions: 2000
MiniLM predictions: 2000


In [40]:
print("MiniLM MAP@3 =", round(sum(mini_scores)/len(mini_scores), 4))

MiniLM MAP@3 = 0.4231


Milestone 2 Q11

Zero-shot classification concepts 

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).


In [41]:
# Milestone 2 Q11

classifier = pipeline("zero-shot-classification")

prompt = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

print(result)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [42]:
# Milestone Q11
print(result["scores"][0])
print(round(result["scores"][0], 4))

0.4574522376060486
0.4575


Milestone 2 Question 12

Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 
What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [43]:
# Milestone 2 Q12

classifier = pipeline("zero-shot-classification")

prompt = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

result_softmax = classifier(
    prompt,
    candidate_labels=candidate_labels
)

print(result_softmax)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [44]:
# Milestone 2 Q12

sum_softmax = sum(result_softmax["scores"])
print(sum_softmax)

0.9999999701976776


In [45]:
# Milestone 2 Q12

result_sigmoid = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

print(result_sigmoid)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

In [46]:
# Milestone 2 Q12

sum_sigmoid = sum(result_sigmoid["scores"])

print(sum_sigmoid)

0.0005095975611766335


In [47]:
# Milestone 2 Q12

difference = abs(sum_softmax - sum_sigmoid)

print(difference)
print(round(difference, 4))

0.999490372636501
0.9995


Milestone 2 Question 13

Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 


In [50]:
# Milestone 2 Q13

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [51]:
# Milestone 2 Q13

row = train_dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(repr(generated))

'B'
